### Parte 1





## Instalar las librerias necesarias

1.   Transformes , datasets , Hugginface

Para este trabajo se utilizará un modelo pre-entrenado para evaluar el sentimiento en frases del sector financiero.

In [2]:
!pip install transformers datasets evaluate huggingface_hub
!pip install -U datasets

In [5]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

# PASOS
dataset = load_dataset("financial_phrasebank", "sentences_allagree")


# Usar el modelo finBERT de HUGGINfaace
model_name = "ProsusAI/finbert"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
nlp_pipeline = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)




Device set to use cpu


#Data de entrenamiento



In [16]:


# Ruta completa al archivo
file_path = '/content/Sentences_AllAgree.txt'

# Leer el archivo con un encoding específico
with open(file_path, 'r', encoding='ISO-8859-1') as file:
    frases = file.readlines()

# Limpiar las frases (eliminar saltos de línea y espacios extra)
frases = [frase.strip() for frase in frases]
frases = frases[42:69] # Aqui puedes cambiar el origen y cantidad de frases
analisis = [nlp_pipeline(frase)[0] for frase in frases]
label_map = {
    "positive": ("✔️", "POSITIVO"),
    "negative": ("❌", "NEGATIVO"),
    "neutral":  ("🤍", "NEUTRO")
}
# Mostrar resultados con estilo
for i in range(10):
    frase = frases[i]
    resultado = analisis[i]
    label = resultado["label"].lower()
    score = resultado["score"] * 100

    emoji, texto = label_map.get(label, ("❓", "DESCONOCIDO"))

    print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
    print(f"📄 *Frase*       : {frase}")
    print(f"{emoji} *Sentimiento* : {texto} ({score:.2f}%)")

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📄 *Frase*       : In January-September 2007 , Finnlines ' net sales rose to EUR 505.4 mn from EUR 473.5 mn in the corresponding period in 2006 .@positive
✔️ *Sentimiento* : POSITIVO (95.15%)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📄 *Frase*       : Adjusted for changes in the Group structure , the Division 's net sales increased by 1.7 % .@positive
✔️ *Sentimiento* : POSITIVO (95.40%)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📄 *Frase*       : 4 February 2011 - Finnish broadband data communication systems provider Teleste Oyj HEL : TLT1V saw its net profit jump to EUR2 .1 m for the last quarter of 2010 from EUR995 ,000 for the same period of 2009 .@positive
✔️ *Sentimiento* : POSITIVO (94.40%)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📄 *Frase*       : Finnish Aktia Group 's operating profit rose to EUR 17.5 mn in the first quarter of 2010 from EUR 8.2 mn in the first quarter of 2009 .@posit

# Parte 2


Instalar la libreria de PyMuPDF

In [17]:
!pip install PyMuPDF

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 23.5 MB/s eta 0:00:00


In [18]:
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Device set to use cpu


In [21]:
import fitz  # PyMuPDF
ruta_pdf = "/content/Contrato_ejercicio.pdf"

def extraer_texto_pdf(ruta_pdf):
    doc = fitz.open(ruta_pdf)
    texto = ""
    for pagina in doc:
        texto += pagina.get_text()
    doc.close()
    return texto

contrato = extraer_texto_pdf(ruta_pdf)

# Dividir el texto en fragmentos de 1000 caracteres (el límite de token del modelo BART)
fragmentos = [contrato[i:i+1000] for i in range(0, len(contrato), 1000)]


1.1 Realizar los resumenes por cada framento

In [25]:
# Generar resumen para cada fragmento
resumenes = []
for fragmento in fragmentos:
    resumen = summarizer(fragmento, max_length=350, min_length=50, do_sample=False)
    resumenes.append(resumen[0]['summary_text'])
# Unir los resúmenes de cada fragmento con un salto de línea entre ellos
resumen_completo = "\n\n".join(resumenes)

Your max_length is set to 350, but your input_length is only 314. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=157)


In [26]:
# Resumen del contrato
print("RESUMEN EJECUTIVO DEL CONTRATO:")
print(resumen_completo)

RESUMEN EJECUTIVO DEL CONTRATO:
Arrendamiento o Alquiler del Local                 Comercial que celebran, de una parte ZAMORA QUISPE, MARÍA ELENA, y de la otra parte, el señor VELÁSQUEZ MENDOZA, JULIO.

La Alameda del Norte N.º 451, en buen estado de conservación y habitabilidad, con piso de cerámica tipo porcelanato. El presente contrato, LA ARRENDADORA se obliga a ceder el uso del local comercial, concerniente como local comerscial, a favor of EL  ARRENDATARIO, a título de alquiler. El monto de la renta pactada en la cláusula siguiente.

La renta será pago por mensualidades, realizándose el abono 11 de cada mes. El comprobante de pago deberá ser enviado por correo electrónico o entregado a LA ARRENDADORA dentro of las 24 horas posteriores a cada depósito.

El Código Civil está obligado a pagar puntualmente el monto de todos los servicios inherentes al bien tales como agua y luz eléctrica, parques y jardines with el impuesto del Patrimonio. Así mismo, EL ARRENDATARIO esta obligado to

## Una nueva utilidad de huggingface Análisis de entidades (NER - Named Entity Recognition)



In [35]:
# Cargar modelo NER
ner_pipeline = pipeline("ner", model="mrm8488/bert-spanish-cased-finetuned-ner", tokenizer="mrm8488/bert-spanish-cased-finetuned-ner", aggregation_strategy="simple", device=-1)


Some weights of the model checkpoint at mrm8488/bert-spanish-cased-finetuned-ner were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


In [36]:
resultados_entidades = ner_pipeline(resumen_completo[:510])


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


In [37]:
# Aplicar el modelo NER al texto limpio

# Procesar las entidades extraídas
def procesar_entidades(entidades):
    entidades_procesadas = {'personas': [], 'lugares': [], 'organizaciones': []}

    for entidad in entidades:
        if entidad.get('score', 0) < 0.7:
            continue
        texto_entidad = entidad['word'].strip()
        tipo_entidad = entidad['entity_group'].upper()

        # Clasificación de las entidades
        if tipo_entidad in ['PER', 'PERSON']:
            if texto_entidad not in entidades_procesadas['personas']:
                entidades_procesadas['personas'].append(texto_entidad)
        elif tipo_entidad in ['LOC', 'LOCATION']:
            if texto_entidad not in entidades_procesadas['lugares']:
                entidades_procesadas['lugares'].append(texto_entidad)
        elif tipo_entidad in ['ORG', 'ORGANIZATION']:
            if texto_entidad not in entidades_procesadas['organizaciones']:
                entidades_procesadas['organizaciones'].append(texto_entidad)
    return entidades_procesadas

# Procesar las entidades
entidades_procesadas = procesar_entidades(resultados_entidades)

# Mostrar un resumen básico de las entidades extraídas
print("🔍 Resumen básico del contrato:")
print(f"Personas identificadas: {entidades_procesadas['personas']}")
print(f"Lugares identificados: {entidades_procesadas['lugares']}")
print(f"Organizaciones identificadas: {entidades_procesadas['organizaciones']}")

🔍 Resumen básico del contrato:
Personas identificadas: ['ZAMORA QUISPE', 'VELÁSQUEZ MENDOZA']
Lugares identificados: ['La Alameda del Norte N']
Organizaciones identificadas: ['LA ARRENDADORA', 'EL ARRENDATARIO']
